In [1]:
!pip install konlpy
!pip install googletrans==4.0.0-rc1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.6/488.6 kB 16.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.4 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17397 sha256=4b82176d14d59cac3ca2aa42b68d4fdd4e49f2e806795d2b15c1e0ef61e2131a
  Stored in directory: /root/.cache/pip/wheels/c0/59/9f/7372f0cf70160fe61b528532e1a7c8498

In [2]:
import json
from google.colab import drive
from konlpy.tag import Okt
from googletrans import Translator

In [2]:
# Google Drive와 연결
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 파일 경로 설정
input_file_path = '/content/drive/My Drive/your_folder_name/raw_conversation.jsonl'  # JSONL 파일 경로
output_file_path = '/content/drive/My Drive/your_folder_name/code_switched_dataset.jsonl'  # 결과 저장 경로

In [ ]:
# 형태소 분석기와 번역기 초기화
okt = Okt()
translator = Translator()

# 자립명사를 추출하는 함수
def extract_independent_nouns(sentence):
    tokens = okt.pos(sentence)
    nouns = [word for word, pos in tokens if pos == "Noun"]
    return nouns

# 영어로 번역하는 함수
def translate_to_english(nouns):
    translations = {}
    for noun in nouns:
        try:
            translations[noun] = translator.translate(noun, src="ko", dest="en").text
        except Exception as e:
            translations[noun] = noun  # 번역 실패 시 원문 유지
    return translations

# 코드스위칭 문장을 생성하는 함수
def code_switch_sentence(sentence, translations):
    code_switched = sentence
    for noun, translation in translations.items():
        code_switched = code_switched.replace(noun, translation)
    return code_switched

# 데이터셋 처리 함수
def process_dataset(data):
    processed_data = []
    for item in data:
        # 원본 데이터 복사
        original_instruction = item.get("instruction", "")
        original_output = item.get("output", "")

        # 자립명사 추출 및 번역
        instruction_nouns = extract_independent_nouns(original_instruction)
        output_nouns = extract_independent_nouns(original_output)
        instruction_translations = translate_to_english(instruction_nouns)
        output_translations = translate_to_english(output_nouns)

        # 코드스위칭된 문장 생성
        code_switched_instruction = code_switch_sentence(original_instruction, instruction_translations)
        code_switched_output = code_switch_sentence(original_output, output_translations)

        # 원본 데이터와 코드스위칭 데이터 병합
        processed_data.append({
            "id": item["id"],
            "original_instruction": original_instruction,
            "original_output": original_output,
            "code_switched_instruction": code_switched_instruction,
            "code_switched_output": code_switched_output,
        })
    return processed_data

In [ ]:
# JSONL 파일 로드
data = []
with open(input_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data.append(json.loads(line))

# 데이터 처리
processed_data = process_dataset(data)

# 처리된 데이터 저장
with open(output_file_path, 'w', encoding='utf-8') as file:
    for item in processed_data:
        json.dump(item, file, ensure_ascii=False)
        file.write('\n')

print(f"Processed dataset saved to {output_file_path}")